# 01 — RealCause TarNet Training

Trains a TarNet generative model on the Sepsis dataset using **identical seed
and hyperparameters** to the original `02_cdv_modeling.ipynb`.

Saves the trained checkpoint to `artifacts/realcause_model/medium_seed_420/model.pt`.

**Run once.** The experiment notebook (02) loads this checkpoint and never retrains.

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import json
import warnings
warnings.filterwarnings('ignore')

from data.sepsis import load_sepsis
from cdv_utils.generator_validation import (
    train_multiple_models, analyze_model_performance, select_best_model
)

ARTIFACTS_DIR = 'cdv_experiments/sepsis/artifacts'
DATASET_PATH  = os.path.join(ARTIFACTS_DIR, 'sepsis_cases.csv')
SAVEROOT      = os.path.join(ARTIFACTS_DIR, 'realcause_model')

assert os.path.exists(DATASET_PATH), (
    f'sepsis_cases.csv not found at {DATASET_PATH}. '
    'Run 00_data_preparation.ipynb first.'
)

os.makedirs(SAVEROOT, exist_ok=True)
print(f'Dataset: {DATASET_PATH}')
print(f'Model will be saved to: {SAVEROOT}')

## 1. Load Data

In [ ]:
# Identical to original NB02: load from artifacts/
sepsis_data = load_sepsis(data_format='numpy', dataroot=ARTIFACTS_DIR)
w, t, y = sepsis_data

original_df = pd.read_csv(DATASET_PATH)
w_cols = [c for c in original_df.columns if c not in ['t', 'y', 'y0', 'y1', 'ite', 'variant']]

print(f'Covariates (w): {w.shape}')
print(f'Treatment (t): {t.shape}, mean={np.mean(t):.3f}')
print(f'Outcome (y): {y.shape}')

## 2. Train TarNet (same seed=420, same architecture as original NB02)

In [ ]:
# Identical hyperparameters to original NB02 Section 3.1
INITIAL_SEED   = 420
training_seeds = [INITIAL_SEED]

print(f'Training RealCause model (seed={INITIAL_SEED})...')
print('Architecture: medium (3 layers, 128 hidden units, ReLU)')
print('Splits: train=50%, val=10%, test=40%')

results, best_model_selector_dict = train_multiple_models(
    w, t, y,
    w_cols=w_cols,
    saveroot=SAVEROOT,
    seeds=training_seeds
)

print('\nTraining complete!')

## 3. Performance Metrics

In [ ]:
performance_df = analyze_model_performance(results, SAVEROOT)
print('Model performance metrics:')
display(performance_df)

best_model = select_best_model(results, best_model_selector_dict, criterion='medium')
print(f'\nSelected model: {best_model.__class__.__name__}')
print(f'Checkpoint: {SAVEROOT}/medium_seed_420/model.pt')

## 4. Quick Sanity Check — ATE and Distributions

In [ ]:
ate = best_model.ate().item()
noisy_ate = best_model.noisy_ate(seed=INITIAL_SEED).item()
print(f'Model ATE (deterministic): {ate:.4f}')
print(f'Model noisy ATE (sampled): {noisy_ate:.4f}')

# Verify checkpoint exists
ckpt = os.path.join(SAVEROOT, 'medium_seed_420', 'model.pt')
assert os.path.exists(ckpt), f'Checkpoint not found: {ckpt}'
print(f'\nCheckpoint confirmed: {ckpt}')
print('Ready for experiment notebook (02_experiment.ipynb).')